In [13]:
import pandas as pd
import numpy as np

In [14]:
df_ledger=pd.read_csv('ledger.csv')
df_ledger.head()

,merchant_id,date,amount,type,type_order,signed_amount,outstanding_balance
0,100274288,2023-11-20,50000.00,drawdown,0,50000.00,50000.00
1,100274288,2023-11-21,5.34,repayment,1,-5.34,49994.66
2,100274288,2023-11-22,49000.00,drawdown,0,49000.00,98994.66
3,100274288,2023-11-24,101000.00,drawdown,0,101000.00,199994.66
4,100274288,2023-11-24,5.00,repayment,1,-5.00,199989.66


1.Number of drawdowns

In [15]:
num_drawdowns_per_customer = df_ledger[df_ledger['type'] == 'drawdown'] .groupby('merchant_id').size().rename('num_drawdowns')
num_drawdowns_per_customer=num_drawdowns_per_customer.sort_index()
num_drawdowns_per_customer.head(10)


merchant_id
100274288      5
100278695      2
107648451     22
109767281      1
109849158     38
109904192      5
109996153     24
110028619     64
111447347      5
111643196    166
Name: num_drawdowns, dtype: int64

2. Repayments within 1 day
3. Repayments within 5 days
4. Repayments after 5 days

In [16]:
df_drawdown = df_ledger[df_ledger['type'] == 'drawdown'][['merchant_id', 'date','amount']].rename(columns={'date':'drawdown_date','amount':'drawdown_amount'})
df_repayment = df_ledger[df_ledger['type'] == 'repayment'][['merchant_id', 'date','amount']].rename(columns={'date':'repaid_date','amount':'repaid_amount'})
df_drawdown['drawdown_date'] = pd.to_datetime(df_drawdown['drawdown_date'])
df_repayment['repaid_date'] = pd.to_datetime(df_repayment['repaid_date'])

drawdowns = df_drawdown.sort_values(['merchant_id', 'drawdown_date'])

# next drawdown date per merchant
drawdowns['next_drawdown_date'] = drawdowns.groupby('merchant_id')['drawdown_date'].shift(-1)

# unique id per drawdown
drawdowns['drawdown_id'] = drawdowns.index


df = drawdowns.merge(
    df_repayment,
    on='merchant_id',
    how='left'
)

# keep only repayments AFTER drawdown
df = df[df['repaid_date'] >= df['drawdown_date']]

# and BEFORE next drawdown (or keep all if last drawdown)
df = df[(df['next_drawdown_date'].isna()) |(df['repaid_date'] < df['next_drawdown_date'])]


df['days_diff'] = (df['repaid_date'] - df['drawdown_date']).dt.days
print(df.head())
repay_1d_amt = df[df['days_diff'] <= 1].groupby(['merchant_id', 'drawdown_id'])['repaid_amount'].sum()
repay_5d_amt = df[(df['days_diff'] <= 5) & (df['days_diff'] > 1) ] .groupby(['merchant_id', 'drawdown_id'])['repaid_amount'].sum()
repay_after_5d_amt = df[df['days_diff'] > 5].groupby(['merchant_id', 'drawdown_id'])['repaid_amount'].sum()
drawdowns = drawdowns.set_index(['merchant_id', 'drawdown_id'])

drawdowns['repay_1d_amt'] = repay_1d_amt
drawdowns['repay_5d_amt'] = repay_5d_amt
drawdowns['repay_after_5d_amt'] = repay_after_5d_amt


drawdowns = drawdowns.fillna(0)


drawdowns['repay_within_1d_flag'] = (
    drawdowns['repay_1d_amt'] / drawdowns['drawdown_amount'] >= 0.5
)

drawdowns['repay_5d_flag'] = (
    drawdowns['repay_5d_amt'] / drawdowns['drawdown_amount'] >= 0.5
)

drawdowns['repay_after_5d_flag'] = (
    drawdowns['repay_after_5d_amt'] / drawdowns['drawdown_amount'] >= 0.5
)


repay_within_1d = drawdowns.groupby('merchant_id')['repay_within_1d_flag'].sum().rename('repay_within_1d')
repay_within_1d=repay_within_1d.sort_index()
print(f"repay_within_1d:{repay_within_1d.head()}")

repay_within_5d = drawdowns.groupby('merchant_id')['repay_5d_flag'].sum().rename('repay_within_5d')
repay_within_5d=repay_within_5d.sort_index()
print(f"repay_within_5d:{repay_within_5d.head()}")


repay_after_5d = drawdowns.groupby('merchant_id')['repay_after_5d_flag'].sum().rename('repay_after_5d')
repay_after_5d=repay_after_5d.sort_index()
print(f"repay_after_5d:{repay_after_5d.head()}")


    merchant_id drawdown_date  drawdown_amount next_drawdown_date  \
0     100274288    2023-11-20          50000.0         2023-11-22   
43    100274288    2023-11-24         101000.0         2023-12-26   
44    100274288    2023-11-24         101000.0         2023-12-26   
45    100274288    2023-11-24         101000.0         2023-12-26   
88    100274288    2023-12-26          20000.0                NaT   

    drawdown_id repaid_date  repaid_amount  days_diff  
0             0  2023-11-21           5.34          1  
43            3  2023-11-24           5.00          0  
44            3  2023-12-24        1398.55         30  
45            3  2023-12-25       50000.00         31  
88            8  2023-12-26       50000.00          0  
repay_within_1d:merchant_id
100274288     1
100278695     1
107648451     8
109767281     1
109849158    20
Name: repay_within_1d, dtype: int64
repay_within_5d:merchant_id
100274288    0
100278695    0
107648451    1
109767281    0
109849158    0
Na

5. Average Monthly Drawdown
6. Average Monthly Repayment
7. Average monthly Balance

In [17]:
df_ledger['date'] = pd.to_datetime(df_ledger['date'])
df_ledger['month'] = df_ledger['date'].dt.to_period('M')

monthly_draw = df_ledger[df_ledger['type'] == 'drawdown'].groupby(['merchant_id', 'month'])['amount'].sum()
avg_monthly_draw = monthly_draw.groupby('merchant_id').mean().rename('avg_monthly_drawdown')
avg_monthly_draw=avg_monthly_draw.sort_index()
print(f"avg_monthly_draw:{avg_monthly_draw.head()}\n")

monthly_repay = df_ledger[df_ledger['type'] == 'repayment'].groupby(['merchant_id', 'month'])['amount'].sum().abs()
avg_monthly_repay = monthly_repay.groupby('merchant_id').mean().rename('avg_monthly_repayment')
avg_monthly_repay=avg_monthly_repay.sort_index()
print(f"avg_monthly_repay:{avg_monthly_repay.head()}\n")


monthly_balance = df_ledger.groupby(['merchant_id', 'month'])['outstanding_balance'].mean()

avg_monthly_balance = monthly_balance.groupby('merchant_id').mean().rename('avg_monthly_pending_balance')
avg_monthly_balance=avg_monthly_balance.sort_index()
print(f"avg_monthly_balance:{avg_monthly_balance.head()}")



avg_monthly_draw:merchant_id
100274288    117000.000000
100278695     16000.000000
107648451    510900.000000
109767281     10000.000000
109849158     43785.363636
Name: avg_monthly_drawdown, dtype: float64

avg_monthly_repay:merchant_id
100274288     18952.076667
100278695      8003.500000
107648451    308288.100000
109767281     11144.600000
109849158     40303.251667
Name: avg_monthly_repayment, dtype: float64

avg_monthly_balance:merchant_id
100274288    83270.517556
100278695     3196.650000
107648451    77268.439798
109767281     2951.933333
109849158    23554.262392
Name: avg_monthly_pending_balance, dtype: float64


In [18]:
# df_ledger['year'] = df_ledger['date'].dt.year
# print(df_ledger['year'].unique())

# yearly_draw = df_ledger[df_ledger['type'] == 'drawdown'].groupby(['merchant_id', 'year'])['amount'].sum()
# print(yearly_draw.head())

# yearly_repay = df_ledger[df_ledger['type'] == 'repayment'].groupby(['merchant_id', 'year'])['amount'].sum().abs()



8. Total Yearly Drawdown
9. Total yearly Repayment

In [19]:
start_date = pd.Timestamp('2023-12-07')
end_date   = pd.Timestamp('2024-12-07')

all_merchants = df_ledger['merchant_id'].unique()
df_window = df_ledger[
    (df_ledger['date'] >= start_date) &
    (df_ledger['date'] <= end_date)
]

total_draw_ttm = df_window[df_window['type'] == 'drawdown'] .groupby('merchant_id')['amount'].sum().reindex(all_merchants, fill_value=0).rename('total_drawdown_ttm')
total_draw_ttm=total_draw_ttm.sort_index()
print(total_draw_ttm.head())

total_repay_ttm = df_window[df_window['type'] == 'repayment'].groupby('merchant_id')['amount'].sum().abs().reindex(all_merchants, fill_value=0).rename('total_repayment_ttm')
total_repay_ttm=total_repay_ttm.sort_index()
print(total_repay_ttm.head())

merchant_id
100274288    34000.0
100278695        0.0
107648451        0.0
109767281        0.0
109849158        0.0
Name: total_drawdown_ttm, dtype: float64
merchant_id
100274288    170558.35
100278695         0.00
107648451         9.00
109767281         0.00
109849158     19391.50
Name: total_repayment_ttm, dtype: float64


Successful Applications

In [25]:
application_df=pd.read_csv('applications (1).csv')
print(application_df.shape)
print(application_df['merchant_id'].nunique())
print (application_df.isna().sum())
filtered_app_df = application_df[
    application_df['merchant_id'].isin(df_ledger['merchant_id'])
]
print(filtered_app_df['merchant_id'].nunique())
# Since Unique customers in application> drawdown, all applications mentioned arent successful. Assuming that the status of application for everyone who has a drawdown is successful
successful_apps = filtered_app_df.groupby('merchant_id').size().reindex(all_merchants, fill_value=0).rename('successful_apps')

print(f"s:{successful_apps.shape}")
successful_apps=successful_apps.sort_index()
print(successful_apps.head())


(398, 2)
370
merchant_id     0
applied_date    0
dtype: int64
345
s:(357,)
merchant_id
100274288    1
100278695    1
107648451    1
109767281    1
109849158    1
Name: successful_apps, dtype: int64


Repayment ratio

In [26]:
repayment_ratio_ttm = (total_repay_ttm / total_draw_ttm).rename('repayment_ratio_ttm')
repayment_ratio_ttm = repayment_ratio_ttm.replace([np.inf, -np.inf], np.nan)
repayment_ratio_ttm = repayment_ratio_ttm.fillna(0)
repayment_ratio_ttm=repayment_ratio_ttm.sort_index()
print(repayment_ratio_ttm.head())

merchant_id
100274288    5.016422
100278695    0.000000
107648451    0.000000
109767281    0.000000
109849158    0.000000
Name: repayment_ratio_ttm, dtype: float64


Average days to repay

In [27]:
df_drawdown = df_drawdown.sort_values(['merchant_id', 'drawdown_date'])

# create unique drawdown_id
df_drawdown['drawdown_id'] = df_drawdown.groupby('merchant_id').cumcount()

# Drawdowns
df_d = df_drawdown.rename(columns={
    'drawdown_date': 'date',
    'drawdown_amount': 'amount'
})
df_d['type'] = 'drawdown'

# Repayments
df_r = df_repayment.rename(columns={
    'repaid_date': 'date',
    'repaid_amount': 'amount'
})
df_r['type'] = 'repayment'
df_r['amount'] = df_r['amount'].abs()  # ensure positive

# Combine
df_ledger = pd.concat([df_d, df_r], ignore_index=True)

# sort properly
df_ledger = df_ledger.sort_values(['merchant_id', 'date']).reset_index(drop=True)

# create row_id for traceability
df_ledger['row_id'] = df_ledger.index

results = []

for merchant, grp in df_ledger.groupby('merchant_id'):
    open_drawdowns = []

    for _, row in grp.iterrows():

        if row['type'] == 'drawdown':
            open_drawdowns.append({
                'drawdown_id': row['drawdown_id'],
                'drawdown_row_id': row['row_id'],
                'date': row['date'],
                'remaining': row['amount']
            })
            
        # repayment    
        else:  
            repay_amt = row['amount']

            while repay_amt > 0 and open_drawdowns:
                dd = open_drawdowns[0]

                alloc = min(dd['remaining'], repay_amt)

                results.append({
                    'merchant_id': merchant,
                    'drawdown_id': dd['drawdown_id'],
                    'drawdown_date': dd['date'],
                    'repayment_date': row['date'],
                    'allocated_amount': alloc
                })

                dd['remaining'] -= alloc
                repay_amt -= alloc

                if dd['remaining'] == 0:
                    open_drawdowns.pop(0)

# Create Allocation DF
df_alloc = pd.DataFrame(results)
print(df_alloc.head())
if not df_alloc.empty:

    df_alloc['days_diff'] = (
        df_alloc['repayment_date'] - df_alloc['drawdown_date']
    ).dt.days
    
    # Compute avg days to repay

    avg_days_to_repay = (
        df_alloc.groupby('merchant_id')
        .apply(lambda x: (x['allocated_amount'] * x['days_diff']).sum() / x['allocated_amount'].sum())
        .rename('avg_days_to_repay')
    )
    

else:
    avg_days_to_repay = pd.DataFrame(columns=['merchant_id', 'avg_days_to_repay'])
avg_days_to_repay=avg_days_to_repay.sort_index()    
avg_days_to_repay.head()

   merchant_id  drawdown_id drawdown_date repayment_date  allocated_amount
0    100274288          0.0    2023-11-20     2023-11-21              5.34
1    100274288          0.0    2023-11-20     2023-11-24              5.00
2    100274288          0.0    2023-11-20     2023-12-24           1398.55
3    100274288          0.0    2023-11-20     2023-12-25          48591.11
4    100274288          1.0    2023-11-22     2023-12-25           1408.89


merchant_id
100274288    47.219484
100278695     0.500000
107648451     2.286265
109767281     0.000880
109849158     7.752774
Name: avg_days_to_repay, dtype: float64

In [28]:
features_df = pd.concat([num_drawdowns_per_customer, repay_within_1d, repay_within_5d,repay_after_5d, avg_monthly_draw, avg_monthly_repay, avg_monthly_balance,
                      total_draw_ttm, total_repay_ttm, successful_apps,repayment_ratio_ttm,avg_days_to_repay], axis=1)#.fillna(0)
features_df.head()

,num_drawdowns,repay_within_1d,repay_within_5d,repay_after_5d,avg_monthly_drawdown,avg_monthly_repayment,avg_monthly_pending_balance,total_drawdown_ttm,total_repayment_ttm,successful_apps,repayment_ratio_ttm,avg_days_to_repay
merchant_id,,,,,,,,,,,,
100274288,5,1,0,2,117000.000000,18952.076667,83270.517556,34000.0,170558.35,1,5.016422,47.219484
100278695,2,1,0,0,16000.000000,8003.500000,3196.650000,0.0,0.00,1,0.000000,0.500000
107648451,22,8,1,0,510900.000000,308288.100000,77268.439798,0.0,9.00,1,0.000000,2.286265
109767281,1,1,0,0,10000.000000,11144.600000,2951.933333,0.0,0.00,1,0.000000,0.000880
109849158,38,20,0,1,43785.363636,40303.251667,23554.262392,0.0,19391.50,1,0.000000,7.752774


In [29]:
features_df.to_csv('features.csv')